In [ ]:
import os
from datetime import datetime
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

# Configuracao de diretorios
BASE_RESULTADO = r'D:\GitHub\DoutoradoCefet\PlanejamentoAnaliseEstatisticoExperimentos\Trabalhos\estudo_dirigido_3\relatorio\resultados'
BASE_DADOS = os.path.join(BASE_RESULTADO, 'dados')
BASE_IMAGENS = os.path.join(BASE_RESULTADO, 'imagens')

METODOS = {
    'analise_completa_meses_anos': {
        'dados': os.path.join(BASE_DADOS, 'analise_completa_meses_anos'),
        'imagens': os.path.join(BASE_IMAGENS, 'analise_completa_meses_anos')
    }
}

for metodo, pastas in METODOS.items():
    for pasta in pastas.values():
        os.makedirs(pasta, exist_ok=True)

def salvar_dados(dados, nome, metodo='analise_completa_meses_anos', tipo='csv'):
    pasta = METODOS[metodo]['dados']
    caminho = os.path.join(pasta, f'{nome}.{tipo}')
    if tipo == 'csv':
        if isinstance(dados, pd.DataFrame):
            dados.to_csv(caminho, index=False)
        else:
            pd.DataFrame(dados).to_csv(caminho, index=False)
    print(f'✅ Salvo: {caminho}')

def salvar_figura(fig, nome, metodo='analise_completa_meses_anos'):
    pasta = METODOS[metodo]['imagens']
    caminho = os.path.join(pasta, f'{nome}.png')
    fig.savefig(caminho, dpi=300, bbox_inches='tight')
    print(f'📊 Figura salva: {caminho}')

nomes_meses = {1: 'Janeiro', 2: 'Fevereiro', 3: 'Marco', 4: 'Abril', 5: 'Maio', 6: 'Junho',
               7: 'Julho', 8: 'Agosto', 9: 'Setembro', 10: 'Outubro', 11: 'Novembro', 12: 'Dezembro'}

print('✅ Configuracao inicializada!')

In [ ]:
# CARREGAMENTO DE DADOS
arquivo_gta = r'D:\OneDrive\Pessoais\Doutorado\Cefet\data\bd_gta_dentro_mg202505091607.csv'
NROWS_SAMPLE = 100000

print('Carregando dados...')
df = pd.read_csv(arquivo_gta, sep=';', nrows=NROWS_SAMPLE, low_memory=False)
print(f'Carregado: {df.shape[0]:,} linhas')

# Processar dados
df['qtd'] = pd.to_numeric(df['qtd'], errors='coerce')
df['dt_emissao_gta'] = pd.to_datetime(df['dt_emissao_gta'], errors='coerce')
df['mes'] = df['dt_emissao_gta'].dt.month
df['ano'] = df['dt_emissao_gta'].dt.year

# Filtrar dados completos (2012-2024)
df_analise = df[(df['ano'] >= 2012) & (df['ano'] <= 2024)].copy()
df_analise = df_analise.dropna(subset=['qtd', 'mes', 'ano'])

print(f'\nDados filtrados (2012-2024):')
print(f'Total registros: {len(df_analise):,}')
print(f'Anos encontrados: {sorted(df_analise["ano"].unique())}')
print(f'Meses: Todos de 1 a 12')
print(f'\nRegistros por ano:')
print(df_analise.groupby('ano').size())

In [ ]:
# PARTE 1: TCL - TEOREMA CENTRAL DO LIMITE
print('\n' + '='*80)
print('PARTE 1: TEOREMA CENTRAL DO LIMITE (TCL)')
print('='*80)

print('\nValidando normalidade das medias amostrais por mes...')
print('(Simulando 10.000 amostras de tamanho n=50 para cada mes)\n')

n_simulacoes = 10000
n_amostra = 50
tcl_resultados = []

fig_tcl, axes_tcl = plt.subplots(3, 4, figsize=(18, 12))
axes_tcl = axes_tcl.flatten()

for mes in range(1, 13):
    dados_mes = df_analise[df_analise['mes'] == mes]['qtd'].values
    
    if len(dados_mes) < 2:
        continue
    
    # Simular distribuicao amostral das medias
    medias_amostrais = []
    for _ in range(n_simulacoes):
        amostra = np.random.choice(dados_mes, size=n_amostra, replace=True)
        medias_amostrais.append(np.mean(amostra))
    
    medias_amostrais = np.array(medias_amostrais)
    
    # Teste de normalidade (Shapiro-Wilk em subset)
    stat_sw, p_val_sw = stats.shapiro(medias_amostrais[::100])
    
    tcl_resultados.append({
        'Mes': mes,
        'Nome': nomes_meses[mes],
        'N_dados': len(dados_mes),
        'Media': np.mean(dados_mes),
        'DP_original': np.std(dados_mes, ddof=1),
        'Media_das_medias': np.mean(medias_amostrais),
        'DP_medias_amostrais': np.std(medias_amostrais, ddof=1),
        'Shapiro_p_value': p_val_sw,
        'Normalidade': 'Sim' if p_val_sw > 0.05 else 'Nao'
    })
    
    # Plot histograma
    ax = axes_tcl[mes - 1]
    ax.hist(medias_amostrais, bins=40, density=True, alpha=0.7, color='skyblue', edgecolor='black')
    
    mu, sigma = np.mean(medias_amostrais), np.std(medias_amostrais)
    x = np.linspace(mu - 4*sigma, mu + 4*sigma, 100)
    ax.plot(x, stats.norm.pdf(x, mu, sigma), 'r-', linewidth=2, label='Normal')
    
    ax.set_title(f'{nomes_meses[mes]}\n(p={p_val_sw:.4f})', fontsize=10, fontweight='bold')
    ax.set_xlabel('Media Amostral', fontsize=9)
    ax.set_ylabel('Densidade', fontsize=9)
    ax.grid(alpha=0.3)

plt.suptitle('TCL: Distribuicao Amostral das Medias por Mes', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
salvar_figura(fig_tcl, '01_TCL_distribuicao_medias_por_mes')

tcl_df = pd.DataFrame(tcl_resultados)
print('\nResultados TCL:')
print(tcl_df[['Mes', 'Nome', 'Media_das_medias', 'DP_medias_amostrais', 'Normalidade']].to_string(index=False))
salvar_dados(tcl_df, '01_TCL_resultados')

print(f'\n✅ TCL VALIDADO: {(tcl_df["Normalidade"] == "Sim").sum()}/12 meses com distribuicao normal!')

In [ ]:
# PARTE 2: IC - INTERVALO DE CONFIANCA
print('\n' + '='*80)
print('PARTE 2: INTERVALO DE CONFIANCA (IC) - 95%')
print('='*80)

ic_resultados = []

for mes in range(1, 13):
    for ano in sorted(df_analise['ano'].unique()):
        dados = df_analise[(df_analise['mes'] == mes) & (df_analise['ano'] == ano)]['qtd'].values
        
        if len(dados) < 2:
            continue
        
        media = np.mean(dados)
        n = len(dados)
        dp = np.std(dados, ddof=1)
        erro_padrao = dp / np.sqrt(n)
        t_critico = stats.t.ppf(0.975, df=n-1)
        margem_erro = t_critico * erro_padrao
        
        ic_resultados.append({
            'Mes': mes,
            'Mes_Nome': nomes_meses[mes],
            'Ano': int(ano),
            'N': n,
            'Media': media,
            'DP': dp,
            'IC_inferior': media - margem_erro,
            'IC_superior': media + margem_erro,
            'Margem_erro': margem_erro
        })

ic_df = pd.DataFrame(ic_resultados)

print(f'Exemplo de IC calculados (Janeiro):') 
print(ic_df[(ic_df['Mes'] == 1)][['Mes_Nome', 'Ano', 'N', 'Media', 'IC_inferior', 'IC_superior']].head(5).to_string(index=False))

salvar_dados(ic_df, '02_IC_completo_mes_ano')

# Visualizacao: IC por mes
fig_ic, axes_ic = plt.subplots(3, 4, figsize=(18, 12))
axes_ic = axes_ic.flatten()

for idx, mes in enumerate(range(1, 13)):
    ax = axes_ic[idx]
    ic_mes = ic_df[ic_df['Mes'] == mes].sort_values('Ano')
    
    x_pos = np.arange(len(ic_mes))
    ax.errorbar(x_pos, ic_mes['Media'], 
               yerr=[ic_mes['Media'] - ic_mes['IC_inferior'], 
                     ic_mes['IC_superior'] - ic_mes['Media']],
               fmt='o', capsize=5, capthick=2, markersize=8,
               color='darkblue', ecolor='red', elinewidth=2, alpha=0.7)
    
    ax.set_xticks(x_pos)
    ax.set_xticklabels([str(int(a)) for a in ic_mes['Ano']], rotation=45, fontsize=8)
    ax.set_ylabel('Quantidade (qtd)', fontsize=9)
    ax.set_title(f'{nomes_meses[mes]}\nIC 95%', fontsize=10, fontweight='bold')
    ax.grid(alpha=0.3, axis='y')

plt.suptitle('IC 95%: Media com Intervalo de Confianca', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
salvar_figura(fig_ic, '02_IC_por_mes_e_ano')

print(f'\n✅ IC CALCULADOS: {len(ic_df)} combinacoes mes-ano')

In [ ]:
# PARTE 3: ANOVA
print('\n' + '='*80)
print('PARTE 3: ANOVA (Analise de Variancia)')
print('='*80)

# ANOVA POR MES
print('\nANOVA - COMPARACAO ENTRE MESES')
print('-' * 80)

grupos_meses = [df_analise[df_analise['mes'] == mes]['qtd'].values for mes in range(1, 13)]
f_stat_mes, p_val_mes = stats.f_oneway(*grupos_meses)

print(f'F-statistic: {f_stat_mes:.4f}')
print(f'p-value: {p_val_mes:.2e}')
print(f'Resultado: {"SIGNIFICANTE" if p_val_mes < 0.05 else "NAO SIGNIFICANTE"}')

# ANOVA POR ANO
print('\nANOVA - COMPARACAO ENTRE ANOS')
print('-' * 80)

anos_lista = sorted(df_analise['ano'].unique())
grupos_anos = [df_analise[df_analise['ano'] == ano]['qtd'].values for ano in anos_lista]
f_stat_ano, p_val_ano = stats.f_oneway(*grupos_anos)

print(f'F-statistic: {f_stat_ano:.4f}')
print(f'p-value: {p_val_ano:.2e}')
print(f'Resultado: {"SIGNIFICANTE" if p_val_ano < 0.05 else "NAO SIGNIFICANTE"}')

# Salvar resultados
anova_results_df = pd.DataFrame({
    'Fator': ['Meses', 'Anos'],
    'F_statistic': [f_stat_mes, f_stat_ano],
    'p_value': [p_val_mes, p_val_ano],
    'Significante': ['Sim' if p_val_mes < 0.05 else 'Nao', 'Sim' if p_val_ano < 0.05 else 'Nao']
})
salvar_dados(anova_results_df, '03_ANOVA_resultados')

print('\n✅ TESTES ANOVA COMPLETOS')

In [ ]:
# VISUALIZACOES COMPARATIVAS
print('\nGerando visualizacoes comparativas...')

fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# Boxplot por mes
ax1 = axes[0, 0]
dados_por_mes = [df_analise[df_analise['mes'] == mes]['qtd'].values for mes in range(1, 13)]
bp = ax1.boxplot(dados_por_mes, labels=[nomes_meses[i][:3] for i in range(1, 13)], patch_artist=True)
for patch in bp['boxes']:
    patch.set_facecolor('lightblue')
ax1.set_ylabel('Quantidade (qtd)', fontsize=11)
ax1.set_title(f'Boxplot por Mes (F={f_stat_mes:.2f})', fontsize=12, fontweight='bold')
ax1.tick_params(axis='x', rotation=45)
ax1.grid(alpha=0.3, axis='y')

# Boxplot por ano
ax2 = axes[0, 1]
dados_por_ano = [df_analise[df_analise['ano'] == ano]['qtd'].values for ano in anos_lista]
bp2 = ax2.boxplot(dados_por_ano, labels=[str(int(ano)) for ano in anos_lista], patch_artist=True)
for patch in bp2['boxes']:
    patch.set_facecolor('lightgreen')
ax2.set_ylabel('Quantidade (qtd)', fontsize=11)
ax2.set_title(f'Boxplot por Ano (F={f_stat_ano:.2f})', fontsize=12, fontweight='bold')
ax2.tick_params(axis='x', rotation=45)
ax2.grid(alpha=0.3, axis='y')

# Heatmap
ax3 = axes[1, 0]
pivot_media = ic_df.pivot_table(values='Media', index='Mes', columns='Ano', aggfunc='mean')
sns.heatmap(pivot_media, annot=True, fmt='.1f', cmap='RdYlGn_r', ax=ax3)
ax3.set_ylabel('Mes', fontsize=11)
ax3.set_title('Heatmap: Media por Mes e Ano', fontsize=12, fontweight='bold')

# Tendencias
ax4 = axes[1, 1]
for mes in range(1, 13):
    dados_mes_ano = ic_df[ic_df['Mes'] == mes].sort_values('Ano')
    ax4.plot(dados_mes_ano['Ano'], dados_mes_ano['Media'], 
            marker='o', label=nomes_meses[mes][:3], alpha=0.7, linewidth=1.5)
ax4.set_ylabel('Media (qtd)', fontsize=11)
ax4.set_xlabel('Ano', fontsize=11)
ax4.set_title('Tendencia: Media por Mes', fontsize=12, fontweight='bold')
ax4.legend(fontsize=7, ncol=2)
ax4.grid(alpha=0.3)

plt.tight_layout()
plt.show()
salvar_figura(fig, '04_comparacao_completa')

In [ ]:
# CONCLUSOES
print('\n' + '='*80)
print('CONCLUSOES FINAIS')
print('='*80)

stats_mes = ic_df.groupby('Mes')['Media'].agg(['mean', 'min', 'max']).reset_index()
mes_max = stats_mes.loc[stats_mes['mean'].idxmax()]
mes_min = stats_mes.loc[stats_mes['mean'].idxmin()]

stats_ano = ic_df.groupby('Ano')['Media'].agg(['mean', 'min', 'max']).reset_index()
ano_max = stats_ano.loc[stats_ano['mean'].idxmax()]
ano_min = stats_ano.loc[stats_ano['mean'].idxmin()]

print(f'\n📊 DADOS: {len(df_analise):,} registros (2012-2024)')
print(f'\n📈 MESES:')
print(f'  Maior media: {int(mes_max["Mes"])} = {mes_max["mean"]:.2f}')
print(f'  Menor media: {int(mes_min["Mes"])} = {mes_min["mean"]:.2f}')
print(f'  Teste ANOVA: F={f_stat_mes:.4f}, p={p_val_mes:.2e}')

print(f'\n📉 ANOS:')
print(f'  Maior media: {int(ano_max["Ano"])} = {ano_max["mean"]:.2f}')
print(f'  Menor media: {int(ano_min["Ano"])} = {ano_min["mean"]:.2f}')
print(f'  Teste ANOVA: F={f_stat_ano:.4f}, p={p_val_ano:.2e}')

print(f'\n✅ VALIDACOES:')
print(f'  TCL: {(tcl_df["Normalidade"] == "Sim").sum()}/12 meses normais')
print(f'  IC: {len(ic_df)} intervalos calculados (95% confianca)')
print(f'  ANOVA: Meses {"significante" if p_val_mes < 0.05 else "NAO significante"}, Anos {"significante" if p_val_ano < 0.05 else "NAO significante"}')

print(f'\n✅ ANALISE COMPLETA FINALIZADA')